In [ ]:

from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM

import torch

type_str = "float32"
torch_type = getattr(torch, type_str)
engine_args = AsyncEngineArgs(
    model="toy_conv",
    dtype=type_str,
    max_model_len=64,
    gpu_memory_utilization=0.05,
    skip_tokenizer_init=True,
    enable_prefix_caching=False,
    #load_format="dummy",
    block_size=128,
    enforce_eager=True,
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=64, skip_sampling=True)


request_id = "1"

# random input:
# T_target x Factor * Freq * Channels
t_target = 10
factor = 4
freq = 80
channels = 256
x = torch.randn(
    t_target, factor * freq * channels,
    dtype=torch_type
)

prompt_len = 1
i = 0
inputs = {
    # dummy tokens
    "prompt_token_ids": [0] * prompt_len,
    # actual inpust to the model in prefill stage
    "custom_inputs": {
        "conv_input": x[i:i+prompt_len]
    }
}
i += prompt_len
outputs = []
async for output in engine.generate(inputs, sampling_params=sampling_params, request_id=request_id):
    conv_output = output.outputs[0].custom_outputs["conv_output"]  # T_target x Factor/2 x Freq/2 x Channels
    print(conv_output.shape)
    outputs.append(conv_output)
    if i >= x.shape[0]:
        break
    inputs = {"conv_input": x[i:i+1]}
    i += 1
    await engine.append_request(request_id=request_id, custom_inputs=inputs)

In [ ]:
output_flat = torch.cat(outputs, dim=0)
print(output_flat.shape)

# T_target x Factor/2 x Freq/2 x Channels
output = output_flat.view(t_target * factor // 4, freq//4, channels)
print(output.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_file

# Configuration
d_model = 256
k_conv = 3

# Create Conv1d
conv2d_first = torch.nn.Conv2d(
    in_channels=d_model,
    out_channels=d_model,
    kernel_size=k_conv,
    stride=2,
    groups=d_model,  # depthwise
    bias=False,
    padding=0,
)
conv2d_second = torch.nn.Conv2d(
    in_channels=d_model,
    out_channels=d_model,
    kernel_size=k_conv,
    stride=2,
    groups=d_model,  # depthwise
    bias=False,
    padding=0,
)



# Load weights
try:
    state_dict = load_file("toy_conv/model.safetensors")
    # reshape weights for conv2d layer
    with torch.no_grad():
        conv2d_first.weight.copy_(state_dict["conv.0.conv_weight"].cuda().flip(0).flip(1).permute(2, 0, 1).unsqueeze(1))
        conv2d_second.weight.copy_(state_dict["conv.1.conv_weight"].cuda().flip(0).flip(1).permute(2, 0, 1).unsqueeze(1))

    print("Weights loaded successfully.")
except Exception as e:
    print(f"Error loading weights: {e}")


# x shape T_target x Factor * Freq * Channels
# input shape should be 1 x channels x Ttarget*factor x freq
t_target = x.shape[0]
factor = 4
freq = 80
channels = 256
x_conv = x.view(t_target * factor, freq, channels).permute(2, 0, 1).unsqueeze(0)
x_conv_padded = torch.nn.functional.pad(x_conv, (3, 0, 3, 0))


print(f"Input shape: {x_conv_padded.shape}")

# Inference
with torch.no_grad():
    y_torch = conv2d_second(conv2d_first(x_conv_padded))

print(f"Output shape: {y_torch.shape}")

y_torch_perm = y_torch.squeeze(0).permute(1, 2, 0)  # 20 x 40 x 256

In [ ]:
# compare vllm and torch outputs
import matplotlib.pyplot as plt


plt.imshow(output.numpy()[:, :, 0].T, aspect="auto")
plt.colorbar()
plt.show()

plt.imshow(y_torch_perm.numpy()[:, :, 0].T, aspect="auto")
plt.colorbar()
plt.show()

plt.imshow(output.numpy()[:, :, 0].T - y_torch_perm.numpy()[:, :, 0].T, aspect="auto")
plt.colorbar()
plt.show()
